# 🎙️ RVC v2 — 내 목소리로 TTS 만들기

> **Google Colab (무료 GPU T4)** 에서 실행하세요.

## 📋 전체 흐름
```
1단계: 환경 설치
2단계: 내 목소리 녹음 파일 업로드
3단계: 전처리 (노이즈 제거 + 슬라이싱)
4단계: 모델 학습 (10~30분)
5단계: 텍스트 → 내 목소리 TTS 생성!
```

## ✅ 사전 준비
- **런타임 → 런타임 유형 변경 → T4 GPU** 설정 필수!
- 목소리 녹음 파일 준비 (WAV/MP3, 5~30분 권장)
- Google Drive 연결 권장 (학습 결과 저장)


---
## 🔧 Step 1: GPU 확인 & 환경 설치

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(f"\n✅ CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU가 없습니다! 런타임 → 런타임 유형 변경 → T4 GPU로 설정하세요.")

In [ ]:
# Google Drive 마운트 (학습 결과 저장용)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 연결 완료")

In [ ]:
# RVC WebUI 클론 & 패키지 설치
import os

if not os.path.exists('/content/Retrieval-based-Voice-Conversion-WebUI'):
    print("📦 RVC 저장소 클론 중...")
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git

%cd /content/Retrieval-based-Voice-Conversion-WebUI

print("📦 필수 패키지 설치 중... (3~5분 소요)")
!pip install -q -r requirements.txt

# 추가 의존성
!pip install -q faiss-gpu==1.7.2 fairseq gradio==3.34.0
!apt-get install -q -y ffmpeg

print("✅ 설치 완료!")

In [ ]:
# 사전학습 모델 다운로드
import os

%cd /content/Retrieval-based-Voice-Conversion-WebUI

os.makedirs('assets/hubert', exist_ok=True)
os.makedirs('assets/rmvpe', exist_ok=True)
os.makedirs('assets/pretrained_v2', exist_ok=True)

print("📥 HuBERT 모델 다운로드 중...")
!wget -q -O assets/hubert/hubert_base.pt \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt

print("📥 RMVPE 피치 모델 다운로드 중...")
!wget -q -O assets/rmvpe/rmvpe.pt \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt

print("📥 사전학습 v2 모델 다운로드 중...")
!wget -q -O assets/pretrained_v2/f0D40k.pth \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth
!wget -q -O assets/pretrained_v2/f0G40k.pth \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth

print("✅ 모델 다운로드 완료!")

---
## 🎤 Step 2: 목소리 녹음 파일 업로드

### 녹음 팁 📌
- **분량**: 최소 5분, 권장 10~30분 (길수록 품질 향상)
- **환경**: 조용한 방, 에코 없는 공간
- **내용**: 다양한 문장을 자연스럽게 읽기
- **형식**: WAV (권장) 또는 MP3
- **샘플레이트**: 44100Hz 또는 48000Hz

> 💡 여러 개의 파일도 OK — 하나의 폴더에 합쳐서 업로드

In [ ]:
# 목소리 파일 업로드
from google.colab import files
import os, shutil

MODEL_NAME = "my_voice"  # ← 원하는 모델 이름으로 변경하세요
DATASET_DIR = f"/content/Retrieval-based-Voice-Conversion-WebUI/dataset/{MODEL_NAME}"
os.makedirs(DATASET_DIR, exist_ok=True)

print("📂 녹음 파일을 업로드하세요 (WAV/MP3)...")
uploaded = files.upload()

for filename, data in uploaded.items():
    dest = os.path.join(DATASET_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  ✅ 저장됨: {filename} ({len(data)/1024/1024:.1f} MB)")

print(f"\n✅ 총 {len(uploaded)}개 파일 업로드 완료")
print(f"📁 저장 경로: {DATASET_DIR}")

---
## ✂️ Step 3: 전처리 (슬라이싱 + 피치 추출)

In [ ]:
# ⚙️ 학습 설정값
MODEL_NAME   = "my_voice"   # Step 2에서 설정한 이름과 동일하게
SAMPLE_RATE  = 40000         # 40000 or 48000
N_CPU        = 4             # CPU 코어 수
DATASET_DIR  = f"/content/Retrieval-based-Voice-Conversion-WebUI/dataset/{MODEL_NAME}"

print(f"모델명: {MODEL_NAME}")
print(f"샘플레이트: {SAMPLE_RATE}")
print(f"데이터셋 경로: {DATASET_DIR}")

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 1. 오디오 슬라이싱 (3~10초 단위로 자르기)
print("✂️ 오디오 슬라이싱 중...")
!python trainset_preprocess_pipeline_print.py \
  "{DATASET_DIR}" {SAMPLE_RATE} {N_CPU} \
  "logs/{MODEL_NAME}" True

print("\n✅ 슬라이싱 완료!")

In [ ]:
# 2. 피처 추출 (HuBERT + 피치)
print("🔍 피처 추출 중... (5~10분 소요)")
!python extract_f0_print.py \
  "logs/{MODEL_NAME}" {N_CPU} "rmvpe"

!python extract_feature_print.py \
  cuda:0 1 0 0 "logs/{MODEL_NAME}" v2

print("\n✅ 피처 추출 완료!")

---
## 🏋️ Step 4: 모델 학습

| 데이터 분량 | 권장 에포크 | 예상 학습 시간 |
|------------|-----------|---------------|
| 5분 이하 | 100~200 | 5~10분 |
| 10~20분 | 200~300 | 15~25분 |
| 30분 이상 | 300~500 | 30~60분 |

In [ ]:
import os

# ⚙️ 학습 하이퍼파라미터
TOTAL_EPOCH     = 200     # ← 데이터 분량에 맞게 조정
BATCH_SIZE      = 7       # T4 GPU 기준 최적값
SAVE_EVERY_EPOCH = 50     # 50 에포크마다 체크포인트 저장

LOG_DIR = f"logs/{MODEL_NAME}"
os.makedirs(LOG_DIR, exist_ok=True)

# filelist 생성
script_lines = [
    'import os',
    f'log_dir    = "{LOG_DIR}"',
    'gt_wav_dir = os.path.join(log_dir, "0_gt_wavs")',
    'feat_dir   = os.path.join(log_dir, "3_feature256")',
    'f0_dir     = os.path.join(log_dir, "2a_f0")',
    'nsf_dir    = os.path.join(log_dir, "2b-f0nsf")',
    'out_path   = os.path.join(log_dir, "filelist.txt")',
    'lines = []',
    'for fname in os.listdir(gt_wav_dir):',
    '    if fname.endswith(".wav"):',
    '        name = fname[:-4]',
    '        wav  = os.path.join(gt_wav_dir, fname)',
    '        feat = os.path.join(feat_dir,   name + ".npy")',
    '        f0   = os.path.join(f0_dir,     name + ".wav.npy")',
    '        nsf  = os.path.join(nsf_dir,    name + ".wav.npy")',
    '        if os.path.exists(feat) and os.path.exists(f0) and os.path.exists(nsf):',
    '            lines.append(wav + "|" + feat + "|" + f0 + "|" + nsf)',
    'with open(out_path, "w") as fp:',
    '    fp.write("\\n".join(lines))',
    'print("filelist 생성 완료:", len(lines), "개 파일")',
]
with open('/tmp/make_filelist.py', 'w') as fp:
    fp.write('\n'.join(script_lines))

!python3 /tmp/make_filelist.py

print(f"\n🏋️ 학습 시작! ({TOTAL_EPOCH} 에포크)")
print("⏳ 학습 중에는 다른 작업을 하지 마세요.")

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI

!python train_nsf_sim_cache_sid_load_pretrain.py \
  -e "{MODEL_NAME}" \
  -sr {SAMPLE_RATE} \
  -f0 1 \
  -bs {BATCH_SIZE} \
  -g 0 \
  -te {TOTAL_EPOCH} \
  -se {SAVE_EVERY_EPOCH} \
  -pg assets/pretrained_v2/f0G40k.pth \
  -pd assets/pretrained_v2/f0D40k.pth \
  -l 1 \
  -c 0 \
  -sw 1 \
  -v v2

print("\n🎉 학습 완료!")

In [ ]:
# FAISS 인덱스 생성 (음색 품질 향상)
print("📊 FAISS 인덱스 생성 중...")

faiss_lines = [
    'import numpy as np, os, faiss',
    f'model_name  = "{MODEL_NAME}"',
    f'log_dir     = "logs/{MODEL_NAME}"',
    'feature_dir = os.path.join(log_dir, "3_feature256")',
    'if os.path.exists(feature_dir):',
    '    npys = []',
    '    for fname in sorted(os.listdir(feature_dir)):',
    '        if fname.endswith(".npy"):',
    '            npys.append(np.load(os.path.join(feature_dir, fname)))',
    '    if npys:',
    '        big_npy = np.concatenate(npys, axis=0)',
    '        n_ivf   = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)',
    '        index   = faiss.index_factory(256, "IVF" + str(n_ivf) + ",Flat")',
    '        index.train(big_npy)',
    '        index.add(big_npy)',
    '        index_path = os.path.join(log_dir, "trained_IVF" + str(n_ivf) + "_Flat_nprobe_1_" + model_name + "_v2.index")',
    '        faiss.write_index(index, index_path)',
    '        print("인덱스 저장:", index_path)',
    '    else:',
    '        print("feature 파일 없음")',
    'else:',
    '    print("feature 디렉토리 없음:", feature_dir)',
]
with open('/tmp/make_faiss.py', 'w') as fp:
    fp.write('\n'.join(faiss_lines))

!python3 /tmp/make_faiss.py
print("✅ 인덱스 생성 완료!")

In [ ]:
# Google Drive에 모델 저장 (선택사항)
import shutil, os

DRIVE_SAVE_PATH = f"/content/drive/MyDrive/RVC_Models/{MODEL_NAME}"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

log_dir = f"logs/{MODEL_NAME}"
saved = []

# 최신 .pth 모델 파일 찾기
for f in os.listdir(log_dir):
    if f.endswith('.pth'):
        shutil.copy(f"{log_dir}/{f}", f"{DRIVE_SAVE_PATH}/{f}")
        saved.append(f)

# 인덱스 파일 저장
for f in os.listdir(log_dir):
    if f.endswith('.index'):
        shutil.copy(f"{log_dir}/{f}", f"{DRIVE_SAVE_PATH}/{f}")
        saved.append(f)

print(f"✅ Google Drive 저장 완료: {DRIVE_SAVE_PATH}")
for s in saved:
    print(f"  📁 {s}")

---
## 🗣️ Step 5: 텍스트 → 내 목소리 TTS 생성!

RVC는 기본적으로 **Voice Conversion** (다른 목소리 → 내 목소리)이므로,  
TTS를 위해 **gTTS 또는 Edge-TTS**로 먼저 기본 음성을 만든 후 → RVC로 내 목소리로 변환합니다.

In [ ]:
# Edge-TTS 설치 (한국어 고품질 기본 음성)
!pip install -q edge-tts
print("✅ Edge-TTS 설치 완료")

In [ ]:
import asyncio, edge_tts, os

# ✏️ 여기에 원하는 텍스트를 입력하세요!
TEXT = "안녕하세요. 이것은 제 목소리로 만든 TTS입니다. 정말 신기하죠?"

# Edge-TTS 한국어 목소리 목록 확인
async def list_korean_voices():
    voices = await edge_tts.list_voices()
    korean = [v for v in voices if v['Locale'].startswith('ko')]
    for v in korean:
        print(f"  {v['ShortName']} - {v['FriendlyName']}")

print("🇰🇷 사용 가능한 한국어 목소리:")
asyncio.run(list_korean_voices())

In [ ]:
import asyncio, edge_tts

# ✏️ 설정
TEXT         = "안녕하세요. 이것은 제 목소리로 만든 TTS입니다. 정말 신기하죠?"  # 변환할 텍스트
BASE_VOICE   = "ko-KR-InJoonNeural"    # 기본 음성 (남성: InJoonNeural, 여성: SunHiNeural)
BASE_OUTPUT  = "/content/base_tts.wav" # 중간 파일

async def generate_base_tts():
    communicate = edge_tts.Communicate(TEXT, BASE_VOICE)
    await communicate.save(BASE_OUTPUT)

print(f"🗣️ 기본 TTS 생성 중...")
print(f"   텍스트: {TEXT}")
asyncio.run(generate_base_tts())
print(f"✅ 기본 음성 생성 완료: {BASE_OUTPUT}")

# 기본 음성 재생
from IPython.display import Audio, display
print("\n🔊 기본 음성 (변환 전):")
display(Audio(BASE_OUTPUT))

In [ ]:
import os, glob

# 학습된 최신 모델 파일 찾기
log_dir = f"logs/{MODEL_NAME}"
pth_files = sorted(glob.glob(f"{log_dir}/*.pth"), key=os.path.getmtime, reverse=True)
index_files = sorted(glob.glob(f"{log_dir}/*.index"), key=os.path.getmtime, reverse=True)

if pth_files:
    MODEL_PATH = pth_files[0]
    print(f"✅ 모델 파일: {MODEL_PATH}")
else:
    print("❌ 학습된 모델 파일이 없습니다. Step 4를 먼저 실행하세요.")

if index_files:
    INDEX_PATH = index_files[0]
    print(f"✅ 인덱스 파일: {INDEX_PATH}")
else:
    INDEX_PATH = ""
    print("⚠️ 인덱스 파일 없음 (품질이 약간 낮을 수 있음)")

In [ ]:
# RVC로 목소리 변환!
import sys
sys.path.append('/content/Retrieval-based-Voice-Conversion-WebUI')

OUTPUT_PATH = "/content/my_voice_tts.wav"

# infer_cli.py 사용
!python infer_cli.py \
  --model_path "{MODEL_PATH}" \
  --index_path "{INDEX_PATH}" \
  --input_path "{BASE_OUTPUT}" \
  --output_path "{OUTPUT_PATH}" \
  --f0method "rmvpe" \
  --f0up_key 0 \
  --index_rate 0.75 \
  --protect 0.33

print(f"\n🎉 변환 완료!")

In [ ]:
# 결과 재생 & 비교
from IPython.display import Audio, display
import os

print("🔊 기본 음성 (변환 전):")
display(Audio(BASE_OUTPUT))

print("\n🎙️ 내 목소리 TTS (변환 후):")
if os.path.exists(OUTPUT_PATH):
    display(Audio(OUTPUT_PATH))
    print(f"\n✅ 파일 저장됨: {OUTPUT_PATH}")
else:
    print("❌ 출력 파일이 생성되지 않았습니다. 오류를 확인하세요.")

In [ ]:
# 결과 파일 다운로드
from google.colab import files
import os

if os.path.exists(OUTPUT_PATH):
    print("📥 TTS 파일 다운로드...")
    files.download(OUTPUT_PATH)
else:
    print("❌ 다운로드할 파일이 없습니다.")

---
## 🔁 반복 사용 — 텍스트만 바꿔서 TTS 생성

학습이 끝난 후에는 아래 셀만 반복 실행하면 됩니다!

In [ ]:
import asyncio, edge_tts, os
from IPython.display import Audio, display

# ✏️ 여기만 수정하세요!
TEXT       = "오늘 날씨가 정말 좋네요. 산책하러 나가볼까요?"  # ← 원하는 텍스트
F0_KEY     = 0     # 피치 조정 (-12 ~ +12, 0=원본)
INDEX_RATE = 0.75  # 음색 유사도 (0~1, 높을수록 내 목소리에 가까움)

BASE_VOICE  = "ko-KR-InJoonNeural"
BASE_OUTPUT = "/content/base_tts.wav"
OUTPUT_PATH = "/content/my_voice_tts.wav"

# 1. 기본 TTS 생성
async def gen_tts():
    await edge_tts.Communicate(TEXT, BASE_VOICE).save(BASE_OUTPUT)
asyncio.run(gen_tts())

# 2. RVC 변환
os.system(f'python infer_cli.py --model_path "{MODEL_PATH}" --index_path "{INDEX_PATH}" --input_path "{BASE_OUTPUT}" --output_path "{OUTPUT_PATH}" --f0method rmvpe --f0up_key {F0_KEY} --index_rate {INDEX_RATE} --protect 0.33')

# 3. 재생
print(f"📢 텍스트: {TEXT}")
print("🎙️ 내 목소리 TTS:")
display(Audio(OUTPUT_PATH))

# 4. 다운로드
from google.colab import files
files.download(OUTPUT_PATH)

---
## 📚 트러블슈팅

| 문제 | 해결 방법 |
|------|----------|
| CUDA out of memory | BATCH_SIZE를 4~5로 줄이기 |
| 목소리가 이상하게 들림 | F0_KEY 조정, 더 많은 데이터 학습 |
| 잡음이 많음 | 녹음 품질 개선, INDEX_RATE 낮추기 |
| 학습이 너무 느림 | 런타임을 T4 GPU로 변경 확인 |
| infer_cli.py 없음 | `!ls` 로 파일 확인, 다른 추론 스크립트 사용 |

## 💡 품질 향상 팁
- 녹음은 **조용한 환경**에서 할수록 좋음
- **다양한 문장** (짧은 것, 긴 것, 의문문, 감탄문 등) 읽기
- 에포크를 늘리면 품질 향상 (단, 과적합 주의)
- **INDEX_RATE** 0.6~0.8이 보통 최적
